### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="diabetes",
    dataset_year="1988",
    domain_str="medical & healthcare",
    # Data Source
    dataset_source="OpenML",
    # Original UCI source is lost, backup on [OpenML](https://www.openml.org/search?type=data&sort=runs&id=37) and [Kaggle](https://www.kaggle.com/datasets/uciml/pima-indians-diabetes-database).
    original_dataset_source_download_link="https://www.kaggle.com/datasets/uciml/pima-indians-diabetes-database",
    download_description="""
kaggle datasets download uciml/pima-indians-diabetes-database -p local-data-warehouse/diabetes/ && unzip local-data-warehouse/diabetes/pima-indians-diabetes-database.zip -d local-data-warehouse/diabetes/ && rm local-data-warehouse/diabetes/pima-indians-diabetes-database.zip && mkdir -p local-data-warehouse/diabetes/ && mv local-data-warehouse/diabetes/diabetes.csv local-data-warehouse/diabetes/input_diabetes.csv
""",
    # References
    academic_reference_bibtex="""@inproceedings{smith1988using,
  title={Using the ADAP learning algorithm to forecast the onset of diabetes mellitus},
  author={Smith, Jack W and Everhart, James E and Dickson, William C and Knowler, William C and Johannes, Robert Scott},
  booktitle={Proceedings of the annual symposium on computer application in medical care},
  pages={261},
  year={1988}
}
""",
    academic_reference_bibtex_key="smith1988using",
    license="CC0: Public Domain",
    data_tags=["IID"],
    curation_comments="""
- We renamed the class variable "Outcome" to "TestedPositiveForDiabetes" and replaced 1 with "Yes", and 0 with "No".
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="TestedPositiveForDiabetes",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="TestedPositiveForDiabetes",
)

## Preprocessing

In [2]:
import pandas as pd

df = pd.read_csv(f"{dataset_mold.path}/input_diabetes.csv")

target_feature = "TestedPositiveForDiabetes"
df = df.rename(columns={"Outcome": target_feature})
df[target_feature] = df[target_feature].replace(1, "Yes").replace(0, "No")

cat_features = [
    "TestedPositiveForDiabetes",
]

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

df[cat_features] = df[cat_features].astype("category")

## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 768
Columns: 9
Use sampling: False (sample size: 768)
Get row duplicates (staged, merged)...
Using top-8 columns for initial filtering: ['DiabetesPedigreeFunction', 'BMI', 'Insulin', 'Glucose', 'Age', 'SkinThickness', 'BloodPressure', 'Pregnancies']
Rows remaining as candidates after top-8 filter: 0 (of 768)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,TestedPositiveForDiabetes
0,6,98,58,33,190,34.0,0.430,43,No
1,2,112,75,32,0,35.7,0.148,21,No
2,2,108,64,0,0,30.8,0.158,21,No
3,8,107,80,0,0,24.6,0.856,34,No
4,7,136,90,0,0,29.9,0.210,50,No


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,TestedPositiveForDiabetes,category,0.0,0.0,2.0,"No, Yes"
1,BMI,float64,0.0,0.0,248.0,"32.0, 31.6, 31.2, 0.0, 33.3, 32.4, 32.9, 30.8, 32.8, 30.1"
2,DiabetesPedigreeFunction,float64,0.0,0.0,517.0,"0.254, 0.258, 0.259, 0.261, 0.238, 0.207, 0.268, 0.284, 0.692, 0.237"
3,Pregnancies,int64,0.0,0.0,17.0,"1, 0, 2, 3, 4, 5, 6, 7, 8, 9"
4,Glucose,int64,0.0,0.0,136.0,"99, 100, 111, 106, 125, 129, 112, 108, 105, 102"
5,BloodPressure,int64,0.0,0.0,47.0,"70, 74, 68, 78, 72, 64, 80, 76, 60, 0"
6,SkinThickness,int64,0.0,0.0,51.0,"0, 32, 30, 27, 23, 28, 33, 18, 31, 19"
7,Insulin,int64,0.0,0.0,186.0,"0, 105, 140, 130, 120, 100, 180, 94, 110, 115"
8,Age,int64,0.0,0.0,52.0,"22, 21, 25, 24, 23, 28, 26, 27, 29, 31"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
Pregnancies,768.0,3.845052,3.369578,0.000,17.00
Glucose,768.0,120.894531,31.972618,0.000,199.00
BloodPressure,768.0,69.105469,19.355807,0.000,122.00
SkinThickness,768.0,20.536458,15.952218,0.000,99.00
Insulin,768.0,79.799479,115.244002,0.000,846.00
BMI,768.0,31.992578,7.884160,0.000,67.10
DiabetesPedigreeFunction,768.0,0.471876,0.331329,0.078,2.42
Age,768.0,33.240885,11.760232,21.000,81.00


In [7]:
# Categorical Feature Statistics
cat_stats

value  count   pct
column                    rank                   
TestedPositiveForDiabetes 1       No    500  65.1
                          2      Yes    268  34.9

In [8]:
# Target Distribution
target_df

,count,pct
TestedPositiveForDiabetes,,
No,500,65.1
Yes,268,34.9


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=10, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

Using Stratified IID splits.


## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to diabetes/019d5a42-a734-7a8e-ab43-f20a26441cdf
019d5a42-a734-7a8e-ab43-f20a26441cdf
67abbb55f163c0f9304213d061b61d15584d8985daaeef6c95c98eff3c03de3d
